# 02 - CPT LoRA training  

This notebook trains Continual Pre-Training (CPT) LoRA adapters on: `cpt_train.jsonl`

In [ ]:
import exp_common as E

RUN_SEEDS = [42, 123, 777]
OVERWRITE = False
MAX_SEQ_LENGTH = 2048
CPT_EPOCHS = 1.0
CPT_MAX_STEPS = -1
BATCH_SIZE = 2
GRAD_ACCUM = 8
LR = 1e-4

rows = E.read_jsonl(E.PREPARED_DIR / "cpt_train.jsonl")
cpt_val = E.read_jsonl(E.PREPARED_DIR / "cpt_val.jsonl")
print("CPT rows:", len(rows), "val:", len(cpt_val))

for seed in RUN_SEEDS:
    out = E.ADAPTERS_DIR / "cpt" / f"seed_{seed}"
    adapter = out / "adapter"
    if adapter.exists() and (adapter / "adapter_model.safetensors").exists() and not OVERWRITE:
        print("Exists, skip:", adapter)
        continue
    E.train_lora_adapter(
        rows, cpt_val, out, seed=seed, max_seq_length=MAX_SEQ_LENGTH,
        epochs=CPT_EPOCHS, max_steps=CPT_MAX_STEPS,
        per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR, lora_r=16, lora_alpha=32, lora_dropout=0.05,
        overwrite=OVERWRITE,
    )
print("CPT training complete")